In [ ]:
import serial
import struct
import json
import time
import os

# --- CONFIGURATION ---
SERIAL_PORT = 'COM6'  # Ensure this matches your device
BAUD_RATE = 9600      # Matches UART5 configuration
DATA_FILE = 'drone_telemetry.json'

def save_to_json(data_list):
    """Helper to save list to JSON file"""
    try:
        with open(DATA_FILE, 'w') as f:
            json.dump(data_list, f, indent=4)
    except Exception as e:
        print(f"Error saving JSON: {e}")

def read_ano_data():
    data_records = []
    buffer = b''
    
    print(f"Connecting to {SERIAL_PORT} to read ANO Protocol data...")
    print("NOTE: This data is sent via UART5. Ensure your wireless module is connected to the UART5 pins (PC12/PD2).")
    
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        print("Connected! Waiting for data stream...")
        
        while True:
            # Read all available data into buffer
            if ser.in_waiting:
                buffer += ser.read(ser.in_waiting)
            
            # The packet ends with the tail: 00 00 80 7f
            # We look for this byte sequence
            tail_seq = b'\x00\x00\x80\x7f'
            tail_idx = buffer.find(tail_seq)
            
            if tail_idx != -1:
                # Found a tail. A full packet is 68 bytes (64 data + 4 tail)
                # Check if we have the preceding 64 bytes in the buffer
                start_idx = tail_idx - 64
                
                if start_idx >= 0:
                    # Extract the full 68-byte packet
                    packet = buffer[start_idx:tail_idx+4]
                    
                    # Remove this packet from the buffer to process the next one
                    buffer = buffer[tail_idx+4:]
                    
                    try:
                        # Unpack 16 floats (64 bytes) from the data section
                        # '<16f' = Little-endian, 16 floats
                        floats = struct.unpack('<16f', packet[:64])
                        
                        # Map the data based on ANO_Report_UserData1 in send_data.c
                        record = {
                            "timestamp": time.time(),
                            "voltage": floats[0],
                            "Z_ratePID_U": floats[1],
                            "locxPID_FB": floats[2],
                            "locyPID_FB": floats[3],
                            "Z_posPID_FB": floats[4],
                            "pitch": floats[5],
                            # Indices 6-15 are dummy values (1, 8, 9...) in the C code
                        }
                        
                        data_records.append(record)
                        
                        # Feedback to user
                        print(f"Captured {len(data_records)} packets. Voltage: {record['voltage']:.2f}V | Pitch: {record['pitch']:.2f}", end='\r')
                        
                        # Save to disk every 50 packets to avoid data loss
                        if len(data_records) % 50 == 0:
                            save_to_json(data_records)
                            
                    except struct.error:
                        print("\nPacket decode error")
                else:
                    # Found a tail, but not enough data before it (incomplete packet)
                    # Discard the buffer up to this tail to resync
                    buffer = buffer[tail_idx+4:]
            
            time.sleep(0.01)

    except serial.SerialException as e:
        print(f"\nError opening serial port: {e}")
    except KeyboardInterrupt:
        print(f"\nStopped by user. Saving {len(data_records)} records...")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
        save_to_json(data_records)
        print(f"Data saved to {os.path.abspath(DATA_FILE)}")

# Run the recorder
read_ano_data()

In [ ]:
import serial
import struct
import time
import json
import datetime

# --- CONFIGURATION ---
SERIAL_PORT = 'COM6' 
BAUD_RATE = 9600
OUTPUT_FILE = 'drone_telemetry.json'

def read_detailed_telemetry():
    print(f"Connecting to {SERIAL_PORT} for detailed telemetry...")
    telemetry_log = []
    
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        print("Connected! Collecting data... (Press Stop to save and exit)")
        
        # New packet size is 68 bytes (16 floats + 4 tail bytes)
        PACKET_SIZE = 68
        
        while True:
            if ser.in_waiting >= PACKET_SIZE:
                # Read full packet
                data = ser.read(PACKET_SIZE)
                
                # Check tail to ensure we are aligned (00 00 80 7f)
                # 0x7f800000 is Infinity in float
                tail = data[64:68]
                if tail != b'\x00\x00\x80\x7f':
                    # If not aligned, read one byte to shift and try again
                    ser.read(1) 
                    continue
                
                try:
                    # Unpack 16 floats
                    # <16f = Little endian, 16 floats
                    values = struct.unpack('<16f', data[0:64])
                    
                    # Map values to names based on our C code change
                    record = {
                        "timestamp": datetime.datetime.now().isoformat(),
                        "voltage": values[0],
                        "pid_z_rate_u": values[1],
                        "pid_locx_fb": values[2],
                        "pid_locy_fb": values[3],
                        "pid_z_pos_fb": values[4],
                        "pitch": values[5],
                        "roll": values[6],
                        "yaw": values[7]
                    }
                    
                    telemetry_log.append(record)
                    
                    # Print status
                    print(f"Volt: {record['voltage']:.2f}V | "
                          f"R/P/Y: {record['roll']:.1f}/{record['pitch']:.1f}/{record['yaw']:.1f} | "
                          f"Captured: {len(telemetry_log)}", end='\r')
                    
                except struct.error:
                    pass
            
            time.sleep(0.005)

    except serial.SerialException as e:
        print(f"\nError: {e}")
    except KeyboardInterrupt:
        print("\nStopping capture...")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
        
        # Save to JSON
        if telemetry_log:
            print(f"\nSaving {len(telemetry_log)} records to {OUTPUT_FILE}...")
            with open(OUTPUT_FILE, 'w') as f:
                json.dump(telemetry_log, f, indent=2)
            print("Done!")
        else:
            print("\nNo data captured.")

read_detailed_telemetry()

In [1]:
import serial
import struct
import time

# --- CONFIGURATION ---
# Based on your screenshot, COM6 is the "USB Serial Device"
SERIAL_PORT = 'COM6' 
BAUD_RATE = 9600  # Matches the configuration in your drone's usart3.c

def read_drone_data():
    print(f"Attempting to connect to {SERIAL_PORT}...")
    try:
        # Open the serial port
        # timeout=1 means it will wait up to 1 second for data before continuing
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        print(f"Successfully connected to {SERIAL_PORT}!")
        print("Waiting for data... (Press Stop/Interrupt to exit)")
        
        # The packet size is 16 bytes as defined in your firmware's usart3_send function
        PACKET_SIZE = 16
        
        while True:
            # Check if we have enough data in the buffer
            if ser.in_waiting >= PACKET_SIZE:
                # Read 16 bytes
                data = ser.read(PACKET_SIZE)
                
                # Unpack the data
                # '<'  : Little-endian (standard for STM32)
                # 'fff': 3 floats (Roll, Pitch, Yaw) - 4 bytes each
                # '4x' : Skip 4 bytes (the tail bytes 00 00 80 7f)
                try:
                    roll, pitch, yaw = struct.unpack('<fff4x', data)
                    
                    # Print the decoded values
                    # \r allows us to overwrite the same line for a clean display
                    print(f"Roll: {roll:6.2f} | Pitch: {pitch:6.2f} | Yaw: {yaw:6.2f}", end='\r')
                    
                except struct.error:
                    print("\nError decoding packet - skipping")
            
            # Small sleep to prevent high CPU usage
            time.sleep(0.01)

    except serial.SerialException as e:
        print(f"\nCould not open {SERIAL_PORT}. Is it in use by another program? Error: {e}")
    except KeyboardInterrupt:
        print("\nStopped by user.")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("\nSerial port closed.")

# Run the function
read_drone_data()

Attempting to connect to COM6...
Successfully connected to COM6!
Waiting for data... (Press Stop/Interrupt to exit)
Roll:   0.00 | Pitch:   0.00 | Yaw:   0.00156462848.00 | Yaw:   0.00aw:   0.008.00 | Yaw:   0.00  0.000
Stopped by user.

Serial port closed.
Roll:   0.00 | Pitch:   0.00 | Yaw:   0.00
Stopped by user.

Serial port closed.
